[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maipatatohoon/bioinsight-360/blob/main/BioInsight_360_Colab_Runbook.ipynb)
[![Repository](https://img.shields.io/badge/source-bioinsight--360-181717?logo=github)](https://github.com/Maipatatohoon/bioinsight-360)
[**Download .ipynb**](https://raw.githubusercontent.com/Maipatatohoon/bioinsight-360/main/BioInsight_360_Colab_Runbook.ipynb)

> This notebook lives in **your repository** (`Maipatatohoon/bioinsight-360`, branch `main`).
> Open it with the badge above so you always get the latest version, and use
> **File → Save a copy in GitHub** if you change it and want the change pushed back.

# BioInsight 360 — Live Backend Runbook (v4)

This notebook installs **R + Bioconductor**, installs the **Python backend**, applies the
method-first/comparison compatibility overlay, starts the **FastAPI server**, opens a
**public HTTPS URL**, starts the **analysis worker**, and runs a **real end-to-end job**.

DESeq2 and edgeR are the two core methods. Each result is uploaded and viewable as soon as
that method finishes; limma-voom remains an optional third view. The DESeq2-vs-edgeR
comparison is a separate, user-triggered calculation over the saved tables.

> ## The one thing you need from this notebook
> **STEP 11 prints a URL like `https://random-words.trycloudflare.com`.**
> Copy it and paste it into your website — STEP 14 explains exactly where.

| | |
|---|---|
| First run | **25–45 min** (mostly R package compilation) |
| Later runs, same runtime | ~2 min |
| GPU | **not needed** — use a CPU runtime |
| Keep the tab open | **yes**. If Colab disconnects, the URL dies. Re-run STEP 10 → 12. |

### How to use this notebook

1. `Runtime → Change runtime type → CPU` (GPU is not used).
2. Run the cells **in order**, top to bottom (▶ or `Ctrl+Enter`).
3. Do **not** use *Run all*: you want to see each step's result and stop at the first ❌.
4. At **STEP 11**, copy the URL immediately. You can re-print it any time with **STEP 14**.
5. If STEP 1 cannot connect, see the box below before changing anything in the notebook.

If STEP 1 says **"Failed to fetch"**, stop and read the box below — it is a browser/network issue, not the notebook.


## ⚠️ STEP 1 says "Failed to fetch"? Read this first

STEP 1 only imports `os`, `platform`, `sys` — it makes **no network calls**. The
message is Colab's browser↔runtime link failing, not this notebook.

1. **Allow third-party cookies for Colab.**
   - Firefox: `about:preferences#privacy` → Enhanced Tracking Protection → **Standard**;
     Manage Exceptions → allow `https://colab.research.google.com` and
     `https://*.googleusercontent.com`; restart Firefox.
   - Chrome: Settings → Privacy and security → Third-party cookies → **Allow third-party
     cookies**; add `colab.research.google.com` to the allowed list.
2. **Turn off content blockers** for Colab (uBlock Origin, Privacy Badger, Ghostery,
   AdGuard), then try a **new private window** and a blank notebook: `print("hi")`.
3. **Change network.** Campus/corporate Wi-Fi often blocks the WebSocket Colab needs.
   Connect the laptop to a **phone hotspot**, open Colab, and run STEP 1. The tunnel
   runs on Colab's VM, so after STEP 11 prints the URL you can switch the laptop back
   to Wi-Fi — the URL stays public. Starting Colab in a phone browser works too.

If `print("hi")` in a brand-new notebook also says "Failed to fetch", the problem is
the network/browser; no notebook change can fix it.

## Step map — what each cell does

| Step | What it does | Time | Must pass? |
|---|---|---|---|
| 1 | Configuration | 1 s | ✅ |
| 2 | System packages + R | 2–4 min | ✅ |
| 3 | BiocManager + `argparse` | 1 min | ✅ |
| 4 | R package **DESeq2** | **10–25 min** | ✅ |
| 5 | R package **edgeR** | 1–3 min | ✅ |
| 6 | R package **limma** (optional view) | 1 min | — |
| 7 | Python packages | 1 min | ✅ |
| 8 | Backend code + method-first compatibility overlay | 10 s | ✅ |
| 9 | Pre-flight check (all installs + new API) | 5 s | ✅ |
| 10 | Start the backend | 10 s | ✅ |
| 11 | **Open the public URL** | 10–30 s | ✅ |
| 12 | Start the worker | 10 s | ✅ |
| 13 | End-to-end test (per-method results + saved comparison) | 1–3 min | ✅ |
| 14 | **Re-print your URL** | 1 s | — |
| 15 | Logs / diagnostics | 1 s | — |
| 16 | *Optional:* thesis validation outputs + ZIP | 1–3 min | — |
| 17 | Stop / restart helpers + troubleshooting | — | — |

**If a step prints ❌, stop and fix that step before continuing.** The troubleshooting table is at the end.


In [ ]:
#@title STEP 1 — Configuration (you normally don't need to change anything) { display-mode: "form" }

BACKEND_REPO  = "https://github.com/Maipatatohoon/bioinsight-backend.git"
FRONTEND_REPO = "Maipatatohoon/bioinsight-360"   # used only for the demo dataset
FRONTEND_BRANCH = "main"

WORKDIR  = "/content/bioinsight-backend"          # backend code lives here
DATA_DIR = "/content/bioinsight_data"             # demo counts + metadata
DB_PATH  = "/content/bioinsight_jobs.db"          # job database
URL_FILE = "/content/BACKEND_URL.txt"             # STEP 11 writes the public URL here

BACKEND_LOG = "/content/backend.log"
WORKER_LOG  = "/content/worker.log"
TUNNEL_LOG  = "/content/cloudflared.log"

WORKER_TOKEN = "dev-token-123"   # must match on backend + worker

# Where your website runs — only used to print convenient links
SITE_URL       = "https://bioinsight-360.vercel.app"
LOCAL_SITE_URL = "http://localhost:3000"

# Set this only if you already have a public backend URL you want to reuse.
MANUAL_BACKEND_URL = ""

import os, platform, sys
os.makedirs(DATA_DIR, exist_ok=True)
print("Python  :", sys.version.split()[0])
print("Machine :", platform.platform())
print("CPUs    :", os.cpu_count())
print("Workdir :", WORKDIR)
print("DB      :", DB_PATH)

## STEP 2 — System packages and R

Installs R 4.x from the CRAN repository (instead of the older Ubuntu copy), plus the C libraries that Bioconductor packages compile against.

If the CRAN repository is unavailable for the runtime's Ubuntu release, the cell falls back to the distro's `r-base` automatically.

In [ ]:
%%bash
set -uo pipefail
export DEBIAN_FRONTEND=noninteractive
. /etc/os-release
CODENAME="${UBUNTU_CODENAME:-${VERSION_CODENAME:-jammy}}"
echo "Ubuntu codename: $CODENAME"

echo ">>> [1/7] apt update"
apt-get update -qq || true

echo ">>> [2/7] base build tools"
apt-get install -y -qq wget ca-certificates gnupg dirmngr software-properties-common \
    build-essential gfortran >/dev/null

echo ">>> [3/7] CRAN apt repository (R 4.x)"
wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc \
  | tee /usr/share/keyrings/cran.gpg >/dev/null
echo "deb [signed-by=/usr/share/keyrings/cran.gpg] https://cloud.r-project.org/bin/linux/ubuntu ${CODENAME}-cran40/" \
  > /etc/apt/sources.list.d/cran.list
if ! apt-get update -qq; then
  echo "    CRAN repository unavailable for ${CODENAME}; falling back to the distro R"
  rm -f /etc/apt/sources.list.d/cran.list
  apt-get update -qq || true
fi

echo ">>> [4/7] r-base + r-base-dev"
apt-get install -y -qq r-base r-base-dev >/dev/null || apt-get install -y -qq r-base >/dev/null

echo ">>> [5/7] system libraries for Bioconductor compilation"
for p in libcurl4-openssl-dev libssl-dev libxml2-dev libglpk-dev \
         libpng-dev libjpeg-dev libtiff-dev libfontconfig1-dev \
         libfreetype6-dev libharfbuzz-dev libfribidi-dev; do
  echo "      - $p"
  apt-get install -y -qq "$p" >/dev/null || echo "        (optional — skipped)"
done

echo ">>> [6/7] versions"
R --version | head -1 || echo "!! R IS NOT INSTALLED"

echo ">>> [7/7] disk"
df -h /content | tail -1
echo "STEP 2 done."

## STEP 3 — BiocManager + argparse

`argparse` is easy to forget: **every R engine in this backend calls `library(argparse)`**, but BiocManager does not install it. Without this cell the engines fail with *"there is no package called 'argparse'"*.

In [ ]:
%%bash
echo ">>> Installing BiocManager (CRAN)"
Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); install.packages("BiocManager", repos = "https://cloud.r-project.org")'

echo ""
echo ">>> Installing argparse (required by all three R engines)"
Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); install.packages("argparse", repos = "https://cloud.r-project.org")'

echo ""
echo ">>> Bioconductor version selected for this R:"
Rscript -e 'cat(as.character(BiocManager::version()), "\n")'

echo ""
Rscript -e 'cat("BiocManager:", requireNamespace("BiocManager", quietly=TRUE), "| argparse:", requireNamespace("argparse", quietly=TRUE), "\n")'

## STEP 4 — DESeq2  *(the long one)*

This is usually **10–25 minutes** because Bioconductor compiles from source. Leave the tab open. When it finishes you should see `DESeq2 installed: TRUE`.

In [ ]:
%%bash
echo ">>> DESeq2 — compiling. This is the slow step; do not stop it."
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("DESeq2", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nDESeq2 installed:", requireNamespace("DESeq2", quietly = TRUE), "\n")'

## STEP 5 — edgeR

In [ ]:
%%bash
echo ">>> edgeR"
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("edgeR", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nedgeR installed:", requireNamespace("edgeR", quietly = TRUE), "\n")'

## STEP 6 — limma

In [ ]:
%%bash
echo ">>> limma"
time Rscript -e 'options(Ncpus = max(1, parallel::detectCores() - 1)); BiocManager::install("limma", ask = FALSE, update = FALSE)'
echo ""
Rscript -e 'cat("\nlimma installed:", requireNamespace("limma", quietly = TRUE), "\n")'

## STEP 7 — Python packages, one by one

In [ ]:
# STEP 7 — Python packages (do NOT upgrade Colab's scientific stack)
import subprocess, sys, importlib

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-input", *args]
    print("$ pip install " + " ".join(args), flush=True)
    r = subprocess.run(cmd, text=True, capture_output=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-8:]
    for line in tail:
        print("   ", line)
    if r.returncode:
        print(f"   pip exit {r.returncode} — continuing; STEP 9 will verify")
    return r.returncode

# Only the packages Colab does not already ship.
pip_install("fastapi", "uvicorn", "python-multipart")

# Reuse Colab's pandas/numpy/scipy/sklearn/pydantic; install only if absent.
for mod, pkg in [("pandas", "pandas"), ("numpy", "numpy"), ("sklearn", "scikit-learn"),
                 ("scipy", "scipy"), ("httpx", "httpx"), ("requests", "requests"),
                 ("pydantic", "pydantic")]:
    try:
        importlib.import_module(mod)
        print(f"OK   {mod} already available")
    except Exception:
        pip_install(pkg)

print("\nImport check:")
import fastapi, uvicorn, pandas, numpy, pydantic, sklearn, scipy, httpx, requests, multipart
for m in (fastapi, uvicorn, pandas, numpy, pydantic, sklearn, scipy, httpx, requests):
    print(f"  {m.__name__:<12} {getattr(m, '__version__', 'ok')}")
print("  multipart    ok")


## STEP 8 — Download the backend code

In [ ]:
# STEP 8 — clone or update bioinsight-backend
import os, subprocess

if os.path.isdir(os.path.join(WORKDIR, ".git")):
    print("Updating existing clone ...")
    subprocess.run(["git", "-C", WORKDIR, "fetch", "--depth", "1", "origin", "main"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "reset", "--hard", "origin/main"], check=False)
else:
    print("Cloning", BACKEND_REPO)
    subprocess.run(["git", "clone", "--depth", "1", BACKEND_REPO, WORKDIR], check=True)

print(subprocess.run(
    ["git", "-C", WORKDIR, "log", "-1", "--format=%h %ad %s", "--date=short"],
    capture_output=True, text=True).stdout.strip())

# --- compatibility overlay: method-first results API ---
# The notebook is self-contained even if the backend repository still points at
# the older all-or-nothing API. This overlay contains only the backend source
# files changed by the method-first/comparison upgrade; no credentials or data.
import base64, io, tarfile, hashlib
UPDATE_B64 = (
    "..."
    "WT7L57f3/H9+cRP7AOABAA=="
)
UPDATE_SHA256 = "d1d34fc95c23178d636545bc8a9f743d8286e8978b4cb1de71603238e6eaa938"
job_manager_py = os.path.join(WORKDIR, "app", "jobs", "job_manager.py")
main_py = os.path.join(WORKDIR, "app", "main.py")

def _read(path):
    try:
        return open(path, encoding="utf-8").read()
    except Exception:
        return ""

needs_update = (
    "update_job_method" not in _read(job_manager_py)
    or "/method/{engine}" not in _read(main_py)
    or not os.path.exists(os.path.join(WORKDIR, "app", "analysis", "comparison.py"))
    or "No result table was uploaded for this method" not in _read(job_manager_py)
    or "condition_contrasts <- contrasts" not in _read(os.path.join(WORKDIR, "app", "edger", "edger_script.R"))
    or "missing values (NaN/blank)" not in _read(os.path.join(WORKDIR, "app", "analysis", "validator.py"))
)

if needs_update:
    blob = base64.b64decode(UPDATE_B64)
    digest = hashlib.sha256(blob).hexdigest()
    print("Applying method-first backend overlay ...")
    print("  sha256:", digest)
    if digest != UPDATE_SHA256:
        raise SystemExit("Overlay checksum mismatch; re-download this notebook.")
    with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as tf:
        tf.extractall(WORKDIR)
    print("Backend overlay applied.")
else:
    print("Method-first backend API already present.")

required = [
    "app/main.py",
    "app/jobs/job_manager.py",
    "app/analysis/comparison.py",
    "workers/colab_worker.py",
    "app/deseq2/deseq2_script.R",
    "app/edger/edger_script.R",
    "app/limma/limma_script.R",
]
missing = [f for f in required if not os.path.exists(os.path.join(WORKDIR, f))]
print("Required files:", "OK" if not missing else f"MISSING {missing}")
print("Per-method API:", "OK" if "update_job_method" in _read(job_manager_py) else "MISSING")


## STEP 9 — Pre-flight check

Verifies **everything** before starting the server: `Rscript`, all four R packages, every Python import, and the backend files. This is the cell that tells you whether STEP 2–8 actually worked.

In [ ]:
# STEP 9 — pre-flight
import os, shutil, subprocess

ok = True

def line(label, passed, detail=""):
    global ok
    ok = ok and passed
    print(f"{'OK  ' if passed else 'FAIL'}  {label:<30} {detail}")

rscript = shutil.which("Rscript")
line("Rscript on PATH", rscript is not None, rscript or "not found")

if rscript:
    ver = subprocess.run(["Rscript", "-e", "cat(R.version.string)"],
                         capture_output=True, text=True).stdout.strip()
    line("R version", bool(ver), ver)

    rcode = ('for (p in c("DESeq2","edgeR","limma","argparse")) '
             'cat(p, requireNamespace(p, quietly=TRUE), "\n")')
    out = subprocess.run(["Rscript", "-e", rcode], capture_output=True, text=True).stdout
    for ln in out.strip().splitlines():
        parts = ln.split()
        if len(parts) == 2:
            pkg_ok = parts[1].upper() == "TRUE"
            # DESeq2 and edgeR are required; limma is an optional third view.
            if parts[0] == "limma" and not pkg_ok:
                print(f"WARN  R package limma             not installed — optional view disabled")
            else:
                line("R package " + parts[0], pkg_ok)

for mod in ["fastapi", "uvicorn", "pandas", "numpy", "multipart",
            "pydantic", "sklearn", "scipy", "httpx", "requests"]:
    try:
        __import__(mod)
        line("python " + mod, True)
    except Exception as e:
        line("python " + mod, False, str(e)[:60])

for f in ["app/main.py", "app/jobs/job_manager.py", "app/analysis/comparison.py",
          "workers/colab_worker.py"]:
    line("file " + f, os.path.exists(os.path.join(WORKDIR, f)))

try:
    manager_src = open(os.path.join(WORKDIR, "app/jobs/job_manager.py"), encoding="utf-8").read()
    main_src = open(os.path.join(WORKDIR, "app/main.py"), encoding="utf-8").read()
    line("per-method job API", "update_job_method" in manager_src)
    line("per-method HTTP routes", "/method/{engine}" in main_src)
    line("saved comparison route", "/comparison" in main_src)
except Exception as e:
    line("method-first API source", False, str(e)[:60])

print()
if ok:
    print("ALL REQUIRED CHECKS PASSED — continue to STEP 10.")
else:
    print("SOMETHING IS MISSING — fix the FAIL lines above first.")
    print("If the API checks failed, re-run STEP 8 (the compatibility overlay).")


## STEP 10 — Start the backend

Starts FastAPI on `localhost:8000` with the Colab runtime and permissive CORS (needed so your deployed site can call the tunnel). Any previous backend process is stopped first, so this cell is safe to re-run.

In [ ]:
# STEP 10 — start the FastAPI backend
import os, subprocess, sys, time
import requests

subprocess.run(["pkill", "-f", "uvicorn app.main:app"], check=False)
time.sleep(1)

env = dict(
    os.environ,
    BIOINSIGHT_WORKER_TOKEN=WORKER_TOKEN,
    BIOINSIGHT_DB_PATH=DB_PATH,
    BIOINSIGHT_ALLOWED_ORIGINS="*",
    BIOINSIGHT_RUNTIME="colab",
    BIOINSIGHT_ENV="development",
    PYTHONUNBUFFERED="1",
)

log = open(BACKEND_LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=WORKDIR, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
)

print("Waiting for http://localhost:8000/health ...")
healthy = False
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.ok:
            print("BACKEND UP:", r.json())
            healthy = True
            break
    except Exception:
        pass

if not healthy:
    print("\nBACKEND DID NOT START. Last 40 log lines:\n")
    print(open(BACKEND_LOG).read()[-4000:])
else:
    print(f"PID {proc.pid} · log {BACKEND_LOG}")

## STEP 11 — Open the public URL  ⬅ **THE URL YOU NEED**

Creates a temporary Cloudflare tunnel (no account needed), waits for the `https://….trycloudflare.com` address, **saves it to `/content/BACKEND_URL.txt`**, and prints it in a box you cannot miss.

> If this cell says *"No tunnel URL"*, wait 20 seconds and run it again. The troubleshooting table is at the end.

In [ ]:
# STEP 11 — public HTTPS tunnel (the URL comes from here)
import os, re, stat, subprocess, time, urllib.request
import requests
from IPython.display import display, HTML

PUBLIC_BACKEND_URL = (MANUAL_BACKEND_URL or "").strip()
CF_BIN = "/content/cloudflared"

def tail(path, n=40):
    try:
        return "\n".join(open(path).read().splitlines()[-n:])
    except Exception:
        return "(no log yet)"

if not PUBLIC_BACKEND_URL:
    # the backend must be running first
    try:
        requests.get("http://localhost:8000/health", timeout=3).raise_for_status()
    except Exception as e:
        print("localhost:8000 is not healthy — run STEP 10 first.")
        print("  ", e)

    subprocess.run(["pkill", "-f", "cloudflared tunnel"], check=False)
    time.sleep(1)

    if not os.path.exists(CF_BIN):
        print("Downloading cloudflared ...")
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            CF_BIN,
        )
        os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IEXEC)

    print("Starting the tunnel ...")
    tlog = open(TUNNEL_LOG, "w")
    subprocess.Popen(
        [CF_BIN, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
        stdout=tlog, stderr=subprocess.STDOUT, start_new_session=True,
    )

    for _ in range(120):
        time.sleep(1)
        try:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
        except Exception:
            m = None
        if m:
            PUBLIC_BACKEND_URL = m.group(0)
            break

if not PUBLIC_BACKEND_URL:
    print("\nNO TUNNEL URL. Last cloudflared log lines:\n")
    print(tail(TUNNEL_LOG))
else:
    open(URL_FILE, "w").write(PUBLIC_BACKEND_URL)

    print("\nChecking the URL from outside Colab ...")
    public_ok = False
    for _ in range(10):
        try:
            r = requests.get(PUBLIC_BACKEND_URL + "/health", timeout=15)
            if r.ok:
                print("PUBLIC HEALTH:", r.json())
                public_ok = True
                break
        except Exception:
            time.sleep(2)

    print("\n" + "=" * 78)
    print("  COPY THIS URL")
    print("")
    print("  " + PUBLIC_BACKEND_URL)
    print("=" * 78)
    print("\nSaved to", URL_FILE, "— re-print it any time with STEP 14.")
    print("\nPaste it into the website (any one of these):")
    print("  live site :", SITE_URL + "/?backend=" + PUBLIC_BACKEND_URL)
    print("  local dev :", LOCAL_SITE_URL + "/?backend=" + PUBLIC_BACKEND_URL)
    print("  manually  : navbar → Backend → paste → Test & Save")

    if not public_ok:
        print("\nWARNING: URL created but the public health check failed.")
        print("Give it 10–20 s, then run STEP 14. If it still fails, see the troubleshooting table.")

    display(HTML(f"""
      <div style="border:2px solid #0284c7;border-radius:12px;padding:16px;margin:12px 0;
                  background:#f0f9ff;font-family:monospace">
        <div style="font-size:13px;color:#0369a1;margin-bottom:6px">
          PUBLIC BACKEND URL — copy this
        </div>
        <div style="font-size:20px;font-weight:bold;word-break:break-all">
          {PUBLIC_BACKEND_URL}
        </div>
      </div>
      <div style="font-family:sans-serif;font-size:14px">
        Open the site with this backend:
        <a href="{SITE_URL}/?backend={PUBLIC_BACKEND_URL}" target="_blank">live site</a> ·
        <a href="{LOCAL_SITE_URL}/?backend={PUBLIC_BACKEND_URL}" target="_blank">local dev</a>
      </div>
    """))

## STEP 12 — Start the worker

The worker polls the backend and runs DESeq2, edgeR and (when available) limma-voom. It runs
**in the background**, so you can keep using the notebook. It talks to `localhost:8000`, not
the tunnel, which is faster and less fragile.

The worker uploads each method's table as soon as that engine finishes. A slow or failed
method does not hide a completed one.

You should see `WORKER REGISTERED`.


In [ ]:
# STEP 12 — start the headless worker (background)
import os, subprocess, sys, time

subprocess.run(["pkill", "-f", "workers/colab_worker.py"], check=False)
time.sleep(1)

env = dict(
    os.environ,
    BIOINSIGHT_BACKEND_URL="http://localhost:8000",
    BIOINSIGHT_WORKER_TOKEN=WORKER_TOKEN,
    PYTHONUNBUFFERED="1",
)

wlog = open(WORKER_LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "workers/colab_worker.py"],
    cwd=WORKDIR, env=env, stdout=wlog, stderr=subprocess.STDOUT, start_new_session=True,
)
time.sleep(6)

tail = open(WORKER_LOG).read()
print("worker PID:", proc.pid, "· log:", WORKER_LOG)
if "Successfully registered" in tail:
    print("WORKER REGISTERED — jobs submitted from the website will now run.")
else:
    print("WORKER NOT CONFIRMED YET. Log so far:\n")
    print(tail[-2000:])
    print("\nMost common cause: the backend is not running (re-run STEP 10).")

## STEP 13 — End-to-end test (proves the whole system works)

Downloads the demo dataset (GSE52778), submits it through the **same API the website uses**,
waits for the job, and then checks each saved method independently. It accepts
`PARTIALLY_COMPLETED` as a valid outcome when exactly one core method succeeded.

When both DESeq2 and edgeR tables exist, the cell also creates the optional comparison and
reads it back to prove it was persisted. The comparison never re-runs R.

The backend renames the grouping column to `condition` before running R, so the request
below sends `design = "~ condition"` even though the CSV column is `group`.

**If this cell ends with "END-TO-END OK", the site's method-first results page will work.**


In [ ]:
# STEP 13 — submit a real job and wait for the result
import json, os, re, time
import pandas as pd
import requests

BASE = "http://localhost:8000"
COUNTS = os.path.join(DATA_DIR, "counts.csv")
META = os.path.join(DATA_DIR, "metadata.csv")
TERMINAL = {"COMPLETED", "PARTIALLY_COMPLETED", "FAILED", "CANCELLED"}

# 1) demo data, with a synthetic fallback so this test never depends on the network
try:
    for name, dest in [("demo_counts.csv", COUNTS), ("demo_metadata.csv", META)]:
        url = f"https://raw.githubusercontent.com/{FRONTEND_REPO}/{FRONTEND_BRANCH}/frontend/public/data/{name}"
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        open(dest, "wb").write(r.content)
    print("Demo data downloaded (GSE52778).")
except Exception as e:
    print("Demo download failed — generating a synthetic dataset instead:", e)
    import numpy as np
    rng = np.random.default_rng(7)
    genes = [f"GENE{i:04d}" for i in range(300)]
    samples = [f"S{i}" for i in range(1, 9)]
    counts = rng.negative_binomial(5, 0.01, size=(len(genes), len(samples)))
    counts[100:150, 4:] *= 4
    pd.DataFrame(counts, index=genes, columns=samples).to_csv(COUNTS)
    pd.DataFrame({"sample": samples, "group": ["CONTROL"] * 4 + ["TREATED"] * 4}).to_csv(META, index=False)

meta = pd.read_csv(META)
group_col = [c for c in meta.columns if c.lower() not in ("sample", "sample_id", "id")][0]
groups = list(dict.fromkeys(meta[group_col].astype(str)))
control = next((g for g in groups if re.search(r"control|ctrl|untreated|baseline|vehicle|wt", g, re.I)), groups[0])
treated = next((g for g in groups if g != control), groups[-1])

params = {
    "control_group": control,
    "treated_group": treated,
    "condition_col": group_col,
    "design": "~ condition",
}
print(f"Submitting: control={control!r}  treated={treated!r}  design=~ condition")

with open(COUNTS, "rb") as cf, open(META, "rb") as mf:
    r = requests.post(
        f"{BASE}/api/analysis",
        files={"counts_file": ("counts.csv", cf, "text/csv"),
               "metadata_file": ("metadata.csv", mf, "text/csv")},
        data={"params": json.dumps(params)},
        timeout=120,
    )

if r.status_code != 202:
    raise SystemExit(f"Submit failed: HTTP {r.status_code}\n{r.text[:1000]}")

job = r.json()["job_id"]
print("job_id:", job)

status, last = None, None
for _ in range(180):                      # up to 15 minutes
    time.sleep(5)
    status = requests.get(f"{BASE}/api/analysis/{job}", timeout=30).json()
    state = f"{status['state']} {status.get('progress', 0)}% — {status.get('message', '')}"
    if state != last:
        print("  ", state)
        last = state
    if status["state"] in TERMINAL:
        break

if not status:
    raise SystemExit("Could not read the job status — is the backend still running? (STEP 15)")

print("\nFinal state:", status["state"])
print("Methods:", json.dumps(status.get("methods", {}), indent=2))
print("Warnings:", status.get("warnings"))

# Per-method endpoints are what the Results tabs use. A method can fail while
# the other one remains fully readable.
results = {}
for engine in ("deseq2", "edger", "limma"):
    method = (status.get("methods") or {}).get(engine, {})
    if method.get("status") == "completed":
        mr = requests.get(f"{BASE}/api/analysis/{job}/method/{engine}", timeout=120)
        if mr.status_code == 200:
            rows = mr.json().get("results") or []
            results[engine] = rows
            sig = [x for x in rows if x.get("padj") is not None and x["padj"] < 0.05]
            print(f"  {engine:<6} HTTP 200 · {len(rows):>4} genes · {len(sig):>3} with padj < 0.05")
        else:
            print(f"  {engine:<6} HTTP {mr.status_code} — {mr.text[:200]}")
    else:
        print(f"  {engine:<6} {method.get('status', 'missing')} — {method.get('error') or 'no error reported'}")

core = [e for e in ("deseq2", "edger") if results.get(e)]
core_failed = [e for e in ("deseq2", "edger") if e not in core]

top = sorted([x for x in (results.get("deseq2") or [])
              if x.get("padj") is not None and x.get("log2FoldChange") is not None],
             key=lambda x: x["padj"])[:3]
if top:
    print("\nTop 3 genes by adjusted p (DESeq2):")
    for x in top:
        print(f"   {x['Gene']:<14} log2FC={x['log2FoldChange']:+.2f}   padj={x['padj']:.3g}")

if len(core) == 2:
    print("\nCreating the optional saved comparison from the persisted tables ...")
    c = requests.post(
        f"{BASE}/api/analysis/{job}/comparison",
        json={"padj_threshold": 0.05, "log2fc_threshold": 1.0},
        timeout=120,
    )
    if c.status_code == 200:
        summary = c.json()["comparison"]["summary"]
        print("Comparison summary:", json.dumps(summary, indent=2))
        saved = requests.get(f"{BASE}/api/analysis/{job}/comparison", timeout=60)
        if saved.status_code == 200 and saved.json().get("comparison", {}).get("summary") == summary:
            print("Saved comparison reloaded successfully (persisted, not recalculated).")
        else:
            print("WARNING: comparison was not readable after saving.")
    else:
        print(f"WARNING: comparison HTTP {c.status_code} — {c.text[:300]}")
    print("\nEND-TO-END OK — DESeq2 and edgeR results are available; comparison is saved.")
    print("limma-voom is optional and does not block these results.")
    print("Next: STEP 14 → copy the URL → Upload → QC → Results → (optional) Comparison.")
elif len(core) == 1:
    print(f"\nPARTIAL END-TO-END: {core[0]} is available; failed/missing core method: {core_failed}.")
    print("The site can show the successful method now. The comparison stays disabled until both exist.")
    print("\n--- worker log (last lines) ---")
    print(open(WORKER_LOG).read()[-3000:])
    print("\n--- backend log (last lines) ---")
    print(open(BACKEND_LOG).read()[-3000:])
else:
    print("\nEND-TO-END FAILED — neither core method produced a table.")
    print("error:", status.get("error"))
    print("\n--- worker log (last lines) ---")
    print(open(WORKER_LOG).read()[-4000:])
    print("\n--- backend log (last lines) ---")
    print(open(BACKEND_LOG).read()[-4000:])


## STEP 14 — Your URL (re-print any time)

Run this whenever you need the address again, for example after scrolling too far up.

In [ ]:
# STEP 14 — re-print the public URL and the exact paste locations
import os
from IPython.display import display, HTML

url = ""
if os.path.exists(URL_FILE):
    url = open(URL_FILE).read().strip()

if not url:
    print("No URL saved yet — run STEP 11 first.")
else:
    print("=" * 78)
    print("  YOUR BACKEND URL")
    print("")
    print("  " + url)
    print("=" * 78)
    print()
    print("Where to paste it — pick ONE:")
    print()
    print("  A) Live site, no redeploy:")
    print("     " + SITE_URL + "/?backend=" + url)
    print()
    print("  B) Local dev server (this machine):")
    print("     " + LOCAL_SITE_URL + "/?backend=" + url)
    print()
    print("  C) Manually: open the site → navbar → Backend → paste → Test & Save")
    print("     Expected result: {\"status\": \"healthy\"}")
    print()
    print("Then: Upload → Load Demo Dataset → Continue → QC → Results")
    print("      DESeq2 and edgeR appear independently; Comparison is optional.")

    display(HTML(f"""
      <div style="font-family:sans-serif">
        <p style="margin:0 0 6px 0;color:#0369a1;font-size:13px">PUBLIC BACKEND URL</p>
        <p style="font-family:monospace;font-size:20px;font-weight:bold;word-break:break-all;
                  margin:0 0 12px 0">{url}</p>
        <p style="margin:0">
          <a href="{SITE_URL}/?backend={url}" target="_blank">Open the live site with this backend</a>
          &nbsp;·&nbsp;
          <a href="{LOCAL_SITE_URL}/?backend={url}" target="_blank">Open local dev</a>
        </p>
      </div>
    """))


## STEP 15 — Logs and diagnostics

A snapshot of all three processes and their last log lines. Re-run it whenever something looks wrong.

In [ ]:
# STEP 15 — status + log snapshot (non-blocking)
import os, subprocess

print("PROCESSES")
out = subprocess.run(
    ["pgrep", "-af", "uvicorn app.main:app|workers/colab_worker.py|cloudflared tunnel"],
    capture_output=True, text=True).stdout.strip()
print(out if out else "  (nothing running — re-run STEP 10 and 12)")

try:
    import requests
    print("\nlocal  /health:", requests.get("http://localhost:8000/health", timeout=5).json())
    if os.path.exists(URL_FILE):
        u = open(URL_FILE).read().strip()
        print("public /health:", requests.get(u + "/health", timeout=15).json())
except Exception as e:
    print("health check failed:", e)

for name, path, n in [("BACKEND", BACKEND_LOG, 25),
                      ("WORKER", WORKER_LOG, 25),
                      ("TUNNEL", TUNNEL_LOG, 12)]:
    print("\n" + "-" * 28, name, "-" * 28)
    try:
        print("\n".join(open(path).read().splitlines()[-n:]))
    except Exception as e:
        print("no log:", e)

## STEP 16 — *Optional:* thesis validation outputs

Runs DESeq2 and edgeR directly on the demo data (no web UI, no worker), saves one CSV per
engine, draws a volcano plot per engine, and gives you a downloadable ZIP. limma-voom is
included when installed. Use this for the thesis results section.

The cell writes a cleaned metadata file with a `condition` column and passes explicit
`--numerator` / `--denominator` levels, matching the backend's contrast direction instead of
parsing a combined contrast string.

You can run this while the backend is running — it uses separate files and does not touch the
job database.


In [ ]:
# STEP 16 — direct R-engine validation outputs + ZIP
import os, re, shutil, subprocess, time
import pandas as pd

OUT = "/content/bioinsight_validation"
os.makedirs(OUT, exist_ok=True)
COUNTS = os.path.join(DATA_DIR, "counts.csv")
META = os.path.join(DATA_DIR, "metadata.csv")
META_CLEAN = os.path.join(OUT, "metadata_condition.csv")

if not (os.path.exists(COUNTS) and os.path.exists(META)):
    raise SystemExit("Demo data missing — run STEP 13 once (it downloads the dataset).")

meta = pd.read_csv(META)
sample_col = next((c for c in meta.columns if c.lower() in ("sample", "sample_id", "id")), None)
if sample_col is None:
    raise SystemExit("Metadata has no sample ID column.")
group_col = [c for c in meta.columns if c != sample_col][0]
groups = list(dict.fromkeys(meta[group_col].astype(str)))
control = next((g for g in groups if re.search(r"control|ctrl|untreated|baseline|vehicle|wt", g, re.I)), groups[0])
treated = next((g for g in groups if g != control), groups[-1])

# Match the backend: sample IDs are row names and the design factor is condition.
clean = meta[[sample_col, group_col]].rename(columns={sample_col: "sample", group_col: "condition"})
clean = clean.dropna(subset=["sample", "condition"]).drop_duplicates(subset=["sample"])
clean.to_csv(META_CLEAN, index=False)
print("Direct-validation contrast:", f"{treated} vs {control}")

scripts = {
    "DESeq2": "app/deseq2/deseq2_script.R",
    "edgeR":  "app/edger/edger_script.R",
    "limma":  "app/limma/limma_script.R",
}

outputs = {}
for name, rel in scripts.items():
    out = os.path.join(OUT, f"{name}_results.csv")
    cmd = ["Rscript", os.path.join(WORKDIR, rel),
           "--counts", COUNTS, "--meta", META_CLEAN, "--out", out,
           "--design", "~ condition",
           "--numerator", str(treated), "--denominator", str(control)]
    print(f"\n=== {name} ===")
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        if name == "limma":
            print("limma is optional and was skipped/failed:\n", (p.stderr or p.stdout)[-1000:])
        else:
            print("FAILED:\n", (p.stderr or p.stdout)[-1500:])
        continue
    df = pd.read_csv(out)
    outputs[name] = df
    sig = df[(df["padj"].notna()) & (df["padj"] < 0.05)]
    fc = sig[(sig["log2FoldChange"].abs() > 1)]
    print(f"  genes tested: {len(df)}  |  padj<0.05: {len(sig)}  |  padj<0.05 & |log2FC|>1: {len(fc)}")
    print(f"  saved {out}  ({time.time()-t0:.1f}s)")

# volcano plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

for name, df in outputs.items():
    d = df.dropna(subset=["padj", "log2FoldChange"])
    if d.empty:
        continue
    x = d["log2FoldChange"]
    y = -np.log10(d["padj"].clip(lower=1e-300))
    sig = (d["padj"] < 0.05) & (d["log2FoldChange"].abs() > 1)
    plt.figure(figsize=(7, 5))
    plt.scatter(x[~sig], y[~sig], s=10, c="#94a3b8", alpha=0.7)
    plt.scatter(x[sig], y[sig], s=14, c="#dc2626", alpha=0.9)
    plt.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.8)
    plt.axvline(1, ls="--", c="grey", lw=0.8)
    plt.axvline(-1, ls="--", c="grey", lw=0.8)
    plt.xlabel("log2 fold change")
    plt.ylabel("-log10 adjusted p")
    plt.title(f"{name}: {treated} vs {control}")
    plt.tight_layout()
    png = os.path.join(OUT, f"{name}_volcano.png")
    plt.savefig(png, dpi=150)
    plt.close()
    print("saved", png)

zip_path = shutil.make_archive("/content/bioinsight_validation", "zip", OUT)
print("\nZIP:", zip_path, f"({os.path.getsize(zip_path)} bytes)")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download it from the Colab file browser:", zip_path)


## STEP 17 — Stop / restart

Run the stop cell when you want a clean slate. Then re-run **STEP 10 → 11 → 12**.

Stop is also useful if you see `Address already in use` or a worker from an old run is picking up jobs.

In [ ]:
# STEP 17 — stop backend + tunnel + worker
import subprocess

for pattern in ("uvicorn app.main:app", "workers/colab_worker.py", "cloudflared tunnel"):
    subprocess.run(["pkill", "-f", pattern], check=False)

print("Stopped backend, worker and tunnel.")
print("The public URL is now dead. Re-run STEP 10 → 11 → 12 to get a new one.")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Rscript: command not found` | STEP 2 failed | Re-run STEP 2, then STEP 9 |
| `there is no package called 'argparse'` | STEP 3 skipped | Run STEP 3 |
| `there is no package called 'DESeq2'` / `edgeR` | STEP 4–5 failed | Re-run the failing step, then STEP 9 |
| `there is no package called 'limma'` | optional STEP 6 skipped | Not fatal. DESeq2/edgeR and the comparison still work. |
| DESeq2 install takes forever | compiling from source, normal | wait; 10–25 min |
| STEP 7 `pip` fails / cell stops | `--upgrade` conflicts with Colab numpy/pandas | use the no-upgrade STEP 7 in this notebook |
| `ImportError: cannot import name '_slice' from 'numpy._core.umath'` | old STEP 7 ran `pip install --upgrade numpy`, mixing numpy files | run `!pip install --no-cache-dir --force-reinstall numpy scipy pandas scikit-learn`, then **Runtime → Restart runtime** (not Factory reset) |
| `ModuleNotFoundError` when starting the backend | STEP 7 not finished | Re-run STEP 7, then STEP 10 |
| `Address already in use` | old backend still running | run STEP 17, then STEP 10 |
| STEP 11 says **No tunnel URL** | cloudflared slow / blocked | wait 20 s, run STEP 11 again; check STEP 15 tunnel log |
| URL works locally but public health fails | tunnel still warming up | wait, run STEP 14; retry STEP 11 |
| Worker says **Failed to register** | backend down, or token mismatch | re-run STEP 10, then STEP 12 |
| Job stays `QUEUED` forever | worker not running | run STEP 12, confirm "WORKER REGISTERED" |
| DESeq2 visible while edgeR is still running | expected method-first behaviour | leave the page open; edgeR appears when uploaded. Do not resubmit. |
| Job `PARTIALLY_COMPLETED` | one core method succeeded and one failed | Open the successful method's tab; read the failed method's error. The comparison is correctly disabled. |
| Comparison button disabled | fewer than two saved core tables | finish both DESeq2 and edgeR; the comparison never runs R itself |
| A gene is significant in both methods but not consensus | the methods disagree on log2FC direction | expected: direction agreement is required |
| `No result was reported by the worker` for DESeq2/edgeR | that engine crashed before uploading, or an incremental upload failed twice | check STEP 15 worker log; the worker falls back to aggregate upload when possible |
| Job `FAILED`: `object 'condition' not found` / `object 'Group' not found` | design/grouping-column mismatch | send `condition_col=<real column>` and `design='~ condition'` (STEP 13 and the frontend do) |
| Upload returns **422** | non-integer/negative counts, duplicate IDs, or missing selected samples | read the validation detail; use raw integer counts and matching sample IDs, ≥2 per group |
| Site says **Analysis Not Found** | browser lost the job ID (new tab / Colab restart) | redo Upload → QC; the job lives in Colab only |
| Colab disconnected | free runtime limit | re-run STEP 1, 9, 10, 11, 12 — the URL changes |
| Website cannot reach the URL | mixed content or old deployment | site and tunnel must both be `https`; or use the navbar Backend box |

### Good to know

- The tunnel URL is **temporary**. It changes every time you restart STEP 11.
- The job database is `/content/bioinsight_jobs.db` and disappears when the Colab runtime is recycled.
- `WORKER_TOKEN` is `dev-token-123`. Fine for a demo; change it before any public release (and set it identically in STEP 1 and the backend).
- Free Colab is enough for the demo (200 genes × 8 samples). The worker is CPU-only.
- DESeq2 and edgeR are the core methods. limma-voom is optional; a limma failure must not hide either core result.
- The comparison is stored on the job. Refreshing the Results page reloads the saved comparison; it recalculates only when you press **Compare**.


## Where exactly to paste the URL

### Option A — fastest (no redeploy)

Open:

```
<SITE_URL>/?backend=<YOUR-TUNNEL-URL>
```

For example:

```
https://bioinsight-360.vercel.app/?backend=https://random-words.trycloudflare.com
```

STEP 14 prints this full link and makes it clickable. The app reads `?backend=`, saves it, and uses it immediately.

### Option B — through the UI

1. Open the website.
2. Click **Backend** in the navbar.
3. Paste the `https://….trycloudflare.com` URL.
4. Click **Test & Save**. You should see `{"status": "healthy"}`.
5. Go to **Upload → Load Demo Dataset → Continue**.

### Option C — local dev server

If you are running the site on this machine:

```
http://localhost:3000/?backend=<YOUR-TUNNEL-URL>
```

or navbar → Backend → paste → Test & Save.

---

### After the URL is connected

```
Upload (demo GSE52778)
   → Continue
   → QC                 (library sizes, detection, PCA, correlation, expression heatmap)
   → Results            (DESeq2 tab and edgeR tab, each as soon as it finishes)
   → Optional Comparison (DESeq2 vs edgeR, from the saved tables; both methods + same direction)
   → Pathways           (g:Profiler enrichment, available when a method result exists)
   → Export             (method CSVs, saved comparison CSV, PDF report)
```

If STEP 13 printed **END-TO-END OK**, both core results and the saved comparison have real data
behind them. If it printed **PARTIAL END-TO-END**, the successful method is still fully usable;
read the failed method's error before resubmitting.
